# Module 5 — Inverse Kinematics
## Unit 6 — Singularities and Solution Selection
### Lesson 6.1 — Singularities: Where IK Breaks Down (Recognition)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
det J = L1*L2*sin(theta2); zero at theta2 = 0 or 180 deg.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

L1,L2=0.4,0.3
def det_jacobian_2link(t1,t2,L1,L2): return L1*L2*np.sin(t2)
for d2 in [90,10,0,180]:
    dj=det_jacobian_2link(0.0,np.radians(d2),L1,L2)
    # cross-check against numpy det
    num=np.linalg.det(jacobian_2link(np.radians(30),np.radians(d2),L1,L2))
    print(f"theta2={d2:4d} deg: det J (formula)={dj:.4f}, numpy det={num:.4f}")


## Coding Exercise (§8)
Verify the det formula; flag near-singular band; it matches pinv-step spikes.


In [ ]:
# YOUR CODE HERE
for d2 in [5,30,90,150,175]:
    f=det_jacobian_2link(0,np.radians(d2),L1,L2)
    n=np.linalg.det(jacobian_2link(np.radians(20),np.radians(d2),L1,L2))
    assert abs(f-n)<1e-9
def near_singular(theta2,eps=0.05): return abs(np.sin(theta2))<eps
assert near_singular(np.radians(2)) and not near_singular(np.radians(90))
print("singularity recognition OK")


## Check your work


In [ ]:
assert abs(det_jacobian_2link(0,0.0,L1,L2))<1e-12          # straight
assert abs(det_jacobian_2link(0,np.pi,L1,L2))<1e-12        # folded
assert det_jacobian_2link(0,np.radians(90),L1,L2)>0.1      # well-conditioned
print("All checks passed.")
